# DriftGate: State-Gated Concept Drift Adaptation in Edge Federated Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1gIarELlF1wwS6I38cAN2DbOCZrki5NW7?usp=sharing)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Adharsh--Arvinth%2Fdriftgate-181717.svg?logo=github&logoColor=white)](https://github.com/Adharsh-Arvinth/driftgate.git)

**Patent Proof-of-Concept & Scientific Replication Notebook**

This notebook demonstrates:
1. Edge concept drift detection via statistical ensembles (ADWIN + PageHinkley + DDM)
2. Sliding-window drift transience classification (`TRANSIENT` vs `PERSISTENT`)
3. Autonomous client 6-state FSM with 0-byte transient muting & layer-divergence selective upload
4. Server-side multi-episode memory aggregation with recurrence penalties
5. Empirical communication efficiency (>70% bandwidth reduction) and convergence comparison vs FedAvg, FedProx, and Oort.


In [ ]:
# Step 1: Self-Contained Environment Setup & Auto-Extraction
!pip install -q numpy scikit-learn matplotlib pandas torch altair
import sys, os, io, zipfile, base64

# Auto-extract embedded repository if running standalone in Colab
if not os.path.exists('src'):
    print('Auto-extracting DriftGate modules into /content/...')
    _b64_data = '...'
    _zip_bytes = base64.b64decode(_b64_data)
    with zipfile.ZipFile(io.BytesIO(_zip_bytes)) as z:
        z.extractall('.')
    print('Extraction complete! Project structure initialized.')

if os.path.exists('src') and 'src' not in sys.path:
    sys.path.insert(0, 'src')

print('Environment initialized successfully. Modules available:', sorted(os.listdir('src'))[:6])


In [ ]:
# Step 2: Run Automated Unit & Integration Tests
!python -m unittest discover tests

In [ ]:
# Step 3: Interactive Live Simulation with DriftGate
from config import get_fast_config
from federation import FederationRunner
import matplotlib.pyplot as plt

# Configure a 15-round simulation with mixed concept drift onset at Round 5
cfg = get_fast_config()
cfg.data.num_clients = 8
cfg.training.num_rounds = 15
cfg.selection.clients_per_round = 4
cfg.drift.global_drift_round = 5
cfg.drift.drift_scenario = 'mixed'
cfg.drift.drift_severity = 0.85

runner = FederationRunner(cfg, seed=42)
results = runner.run()

eval_res = results['final_eval']
comm_res = results['comm_stats']
print(f"\n[Simulation Summary]")
print(f"Final Global Accuracy: {eval_res['accuracy']:.4f}")
print(f"F1 Score:              {eval_res['f1']:.4f}")
print(f"Bandwidth Reduction:   {comm_res['bandwidth_reduction_pct']:.1f}%")
print(f"Suppressed Rounds:     {comm_res['suppressed_client_rounds']}")
print(f"Selective Rounds:      {comm_res['selective_client_rounds']}")

In [ ]:
# Step 4: Visualize FSM State Evolution and Accuracy Curves
import pandas as pd

rounds = [m['round'] for m in results['round_metrics']]
accs = [m['accuracy'] for m in results['round_metrics']]
st_hist = results.get('state_history', [])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy curve
ax1.plot(rounds, accs, marker='o', color='#2563eb', linewidth=2, label='DriftGate Global Model')
ax1.axvline(x=5, color='red', linestyle='--', label='Drift Injected (Round 5)')
ax1.set_title('Global Model Accuracy under Concept Drift')
ax1.set_xlabel('Federation Round')
ax1.set_ylabel('Validation Accuracy')
ax1.grid(True, alpha=0.3)
ax1.legend()

# State counts breakdown
if st_hist:
    df_states = pd.DataFrame(st_hist)
    for col, color in [('stable', '#10b981'), ('drifting', '#f59e0b'), ('transient', '#ef4444'), ('persistent', '#8b5cf6'), ('adapting', '#06b6d4'), ('eligible', '#3b82f6')]:
        if col in df_states.columns:
            ax2.plot(df_states['round'], df_states[col], marker='s', label=col.capitalize(), color=color)
    ax2.set_title('Edge Client 6-State FSM Distribution')
    ax2.set_xlabel('Federation Round')
    ax2.set_ylabel('Client Count')
    ax2.grid(True, alpha=0.3)
    ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Step 5: Execute Full Benchmark & Ablations (Reproducing Figures)
!python run_benchmarks.py

# Display Publication Plots
from IPython.display import Image, display
for plot_path in ['results/accuracy_curves.png', 'results/comm_efficiency.png', 'results/ablation_summary.png', 'results/state_distribution.png']:
    if os.path.exists(plot_path):
        print(f"\nDisplaying: {plot_path}")
        display(Image(plot_path))